# Customer Analytics & Revenue Protection Platform
### Interactive Exploratory Data Analysis & Modeling Pipeline Walkthrough

This notebook demonstrates the end-to-end capabilities of the Customer Analytics & Revenue Protection Platform:
1. **Synthetic Data Generation** simulating banking behavior.
2. **Data Cleaning & Feature Engineering** to extract key customer engagement indexes.
3. **Customer Segmentation** using unsupervised K-Means to identify cohort behaviors.
4. **Churn Prediction** using a tuned XGBoost Classifier.
5. **Explainable AI (XAI)** using SHAP for global and individual model attribution.
6. **Financial Calculations** (CLV and Revenue at Risk).
7. **Actionable Retention Playbooks** generated dynamically.

In [1]:
import os
import sys
# Ensure the src directory is in the path to load our local packages
sys.path.insert(0, os.path.abspath(os.path.join('..')))

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import joblib
import shap

from src.preprocess import generate_synthetic_data, ChurnPreprocessor
from src.segmentation import CustomerSegmenter
from src.clv import calculate_clv
from src.revenue_risk import calculate_customer_revenue_risk, calculate_portfolio_revenue_risk
from src.predict import CustomerInferencePipeline

%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')

## 1. Data Generation & Exploration
Let's generate the synthetic dataset of 5,000 customers containing demographic, transaction, and complaint features.

In [2]:
raw_data = generate_synthetic_data(n_samples=5000, random_state=42)
print(f"Raw dataset dimensions: {raw_data.shape}")
print(f"Overall baseline Churn rate: {raw_data['Churn'].mean():.2%}")
raw_data.head()

Raw dataset dimensions: (5000, 14)
Overall baseline Churn rate: 13.38%


,CustomerID,Age,Gender,Geography,TenureMonths,Balance,NumOfProducts,HasCrCard,IsActiveMember,EstimatedSalary,TransactionVolume,TransactionAmount,Complaints,Churn
0,CUST-10000,46,Male,Mumbai,47,0.000000,3,1,0,3.479522e+05,2,3106.444744,0,0
1,CUST-10001,39,Male,Mumbai,65,362478.992097,2,1,0,2.084309e+06,6,17840.281701,0,0
2,CUST-10002,48,Male,Bengaluru,15,129849.194849,2,1,0,1.276950e+06,5,4121.215005,0,0
3,CUST-10003,59,Male,Mumbai,79,120768.458665,2,1,1,2.163598e+06,9,46044.616273,0,0
4,CUST-10004,38,Male,Mumbai,52,87215.457057,2,1,0,2.007167e+06,6,34007.713247,1,1


## 2. Feature Engineering & Preprocessing
We use our custom `ChurnPreprocessor` to handle category encoding, scale values, and extract features like `BalanceToSalaryRatio` and `EngagementScore`.

In [3]:
preprocessor = ChurnPreprocessor()
X_raw = raw_data.drop(columns=['CustomerID', 'Churn'])
y = raw_data['Churn']

X_processed = preprocessor.fit_transform(X_raw)
print(f"Processed feature space dimensions: {X_processed.shape}")
print("Engineered Columns:", preprocessor.feature_names)

Processed feature space dimensions: (5000, 20)
Engineered Columns: ['Age', 'TenureMonths', 'Balance', 'NumOfProducts', 'EstimatedSalary', 'TransactionVolume', 'TransactionAmount', 'Complaints', 'BalanceToSalaryRatio', 'AverageTransactionSize', 'ComplaintsPerTenure', 'EngagementScore', 'BalancePerProduct', 'HasCrCard', 'IsActiveMember', 'Gender_Female', 'Gender_Male', 'Geography_Bengaluru', 'Geography_Delhi', 'Geography_Mumbai']


## 3. Customer Segmentation (Unsupervised K-Means)
Segment customers into four distinct business cohorts: `Loyal High Savers`, `High Value At Risk`, `Standard Mid-Tier`, and `Low-Value Casuals`.

In [ ]:
segmenter = CustomerSegmenter(n_clusters=4, random_state=42)
segmenter.fit(X_processed, raw_data)
raw_data['Segment'] = segmenter.get_business_segments(X_processed)

segment_profiles = raw_data.groupby('Segment').agg(
    CustomerCount=('CustomerID', 'count'),
    AverageBalance=('Balance', 'mean'),
    AverageComplaints=('Complaints', 'mean'),
    ObservedChurn=('Churn', 'mean')
).round(2)
segment_profiles

## 4. Churn Classification (XGBoost)
Split our data and load the pre-trained model (or fit a new one) to evaluate performance.

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, roc_auc_score
from xgboost import XGBClassifier

X_train, X_val, y_train, y_val = train_test_split(
    X_processed, y, test_size=0.2, random_state=42, stratify=y
)

churn_model = XGBClassifier(eval_metric='logloss', random_state=42)
churn_model.fit(X_train, y_train)

y_prob = churn_model.predict_proba(X_val)[:, 1]
print(f"Validation ROC-AUC: {roc_auc_score(y_val, y_prob):.4f}")

## 5. Model Explainability via SHAP
Let's see what features are overall driving the model's predictions using a SHAP Beeswarm summary plot.

In [ ]:
explainer = shap.TreeExplainer(churn_model)
shap_vals = explainer.shap_values(X_processed)

# Handle shape for binary classification outputs
if isinstance(shap_vals, list) and len(shap_vals) > 1:
    shap_vals = shap_vals[1]
    
plt.figure(figsize=(10, 6))
shap.summary_plot(shap_vals, X_processed, feature_names=preprocessor.feature_names, show=False)
plt.title("Global Churn Feature Importances (SHAP Beeswarm)", fontsize=14, pad=15)
plt.show()

## 6. Business Predictions, Customer Lifetime Value, and Retention Playbooks
Let's test the complete unified inference pipeline on a single high-risk customer profile.

In [ ]:
# Load files through unified inference pipeline
# Note: ensure train.py was run to save models inside parent '../models/' directory
inference = CustomerInferencePipeline(models_dir="../models")

sample_customer = {
    "CustomerID": "CUST-TEST-01",
    "Age": 45,
    "Gender": "Female",
    "Geography": "Mumbai",
    "TenureMonths": 8,
    "Balance": 1500000.0,
    "NumOfProducts": 3,
    "HasCrCard": 1,
    "IsActiveMember": 0,
    "EstimatedSalary": 1000000.0,
    "TransactionVolume": 1,
    "TransactionAmount": 8000.0,
    "Complaints": 3
}

report = inference.analyze_customer(sample_customer)
inference.print_executive_report(report)